# Transcriptome vs 3'UTRome over mouse SVZ neurogenesis

Author: Andre Macedo Date: 2026-07-01 Kernel: `apa-python`

Purpose: Show that the developmental 3'UTR-length programme ("UTRome") carries
information beyond what gene expression already tells us along the adult mouse SVZ
neurogenic lineage (qNSC -> neuroblast -> neuron).

Modelled on Kremer et al. 2023, Fig. 2a (side-by-side correlation matrices +
hierarchical clustering of transcriptome vs methylome across cell states, showing the
methylome separates states the transcriptome merges). Here the second modality is the
per-gene 3'UTR-length index W, computed from within-gene isoform proportions, so it
is decoupled from gene-expression level (proportions remove the gene-count
signal, which would otherwise make a biased bridge back to the expression correlations).

We compare the two modalities three ways:

1. Cell states collapsed to cell types (Kremer Fig. 2a analogue), incl. terminal neurons.
2. Finely binned over pseudotime (equal-frequency / population quantile bins), finer
   than "all neuroblasts", to resolve smaller within-stage changes.
3. Per gene: whether a gene's 3'UTR-length trajectory is decoupled from its expression trajectory.

Panels are saved to `Fig1/unassigned_panels/`. The Spearman pseudotime-bin correlation matrices
(section 5) are used for Fig. 1H; the enrichment of the expression-invisible APA genes (section 10)
is replotted in fig. S1K (`figures/fig1/supp_12_expression_invisible_reactome`). The other panels
are exploratory.


## 1. Setup

Shared paper style (`paper_fig_style`), paths, and analysis constants. Panels are written
with descriptive names to `Fig1/unassigned_panels/` via `S.save_and_show(..., fig_dir=...)`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import subprocess
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import anndata as ad
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import cophenet, dendrogram, leaves_list, linkage
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr, rankdata, spearmanr

warnings.filterwarnings("ignore")

# shared paper style (palette, fonts, Inkscape-editable SVG, save_and_show)
STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

RNG = np.random.default_rng(0)  # reproducible Mantel permutations

# ---- paths -----------------------------------------------------------------
BASE = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis")
UTR_BASE = BASE / "2026_analyses/utrcalling"

LINEAGE_H5 = UTR_BASE / (
    "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_lineage.h5ad"
)
UTR_RAW_H5 = UTR_BASE / "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"
EXPR_H5 = BASE / (
    "cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos.h5ad"
)
RANKING_CSV = UTR_BASE / "DirMult_optimized/composite_ranking_optimized2_df5.csv"

# outputs: exploratory panels + a small results table
UNASSIGNED_DIR = S.OUT_ROOT / "Fig1" / "unassigned_panels"
UNASSIGNED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = UTR_BASE / "correlation_analyses_proportions"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- analysis constants ----------------------------------------------------
CORE_LINEAGE = [
    "qNSC1",
    "qNSC2",
    "aNSC1",
    "aNSC2",
    "TAP",
    "NB",
]  # Fig-1 pseudotime order
CELLTYPE_STATES = CORE_LINEAGE + ["Neurons"]  # + terminal (discrete) neurons
N_BINS = 20  # equal-frequency pseudotime bins over the core lineage
MIN_READS = 25  # min summed UMIs for a (gene, state) to be usable (celltype states)
MIN_READS_BIN = 20  # min summed UMIs per (gene, bin)
TOP_N_FEATURES = 1000  # top-variance genes for the correlation heatmaps (Kremer VMR-style)
N_PERM = 9999  # Mantel permutations

# modality accent colours
COL_TX = "#3B7A57"  # transcriptome (green)
COL_UTR = "#7B3294"  # UTRome (purple)
CORR_CMAP = "magma"

print("style + constants ready")
print("panels ->", UNASSIGNED_DIR)

## 2. Load data and define the analysis cell set

- Lineage h5ad provides the Fig-1 pseudotime (`dpt_pseudotime`), `celltype`, and the
  neurogenic cell set (14,115 cells).
- Raw UTR counts (per PAS site) are subset to those cells; this single matrix gives
  both modalities: gene-total UMIs (expression) and within-gene proportions (UTR length),
  from the same molecules, same cells, same genes (a matched comparison).
- Expression h5ad (full transcriptome, `.raw`) is loaded only for a robustness panel;
  cell IDs are reconciled `BARCODE-BATCH` <-> `BARCODE_BATCH`.

We restrict to the 7 cell-type states (6 core lineage + terminal Neurons). Pseudotime binning
later uses only the 6 core lineage cells with a valid pseudotime (Neurons are a discrete
terminal state, ~all NaN-pseudotime, so they cannot be placed on the trajectory).


In [ ]:
from scipy.sparse import issparse

lineage = ad.read_h5ad(LINEAGE_H5)
raw = ad.read_h5ad(UTR_RAW_H5)
raw = raw[lineage.obs_names].copy()  # align to the lineage cell set + order
assert list(raw.var_names) == list(lineage.var_names)

celltype = lineage.obs["celltype"].astype(str).values
pt = lineage.obs["dpt_pseudotime"].astype(float).values

# keep the 7 cell-type states
keep = np.isin(celltype, CELLTYPE_STATES)
Xutr = raw.X.tocsr() if issparse(raw.X) else raw.X
Xutr = Xutr[keep]
celltype = celltype[keep]
pt = pt[keep]
obs_names = raw.obs_names[keep]

# per-site gene identity (ENSEMBL) + rank-normalised UTR length
site_gene = raw.var["name"].astype(str).values  # ENSEMBL per site
site_sym = raw.var["gene_name"].astype(str).values  # symbol per site
site_size = raw.var["utr_size"].astype(float).values  # PAS distance (bp)

genes = list(dict.fromkeys(site_gene))  # unique ENSEMBL, stable order
g2idx = defaultdict(list)
for i, g in enumerate(site_gene):
    g2idx[g].append(i)
g2idx = {g: np.array(v) for g, v in g2idx.items()}
gene_sym = {g: site_sym[idx[0]] for g, idx in g2idx.items()}

# rank-normalised site ranks per gene:  shortest PAS -> 0, longest -> 1
gene_ranks = {}
for g, idx in g2idx.items():
    K = len(idx)
    r = rankdata(site_size[idx], method="average")
    gene_ranks[g] = (r - 1) / (K - 1) if K > 1 else np.zeros(K)

print(f"cells kept (7 states): {keep.sum()} | sites: {Xutr.shape[1]} | genes: {len(genes)}")
print("cells per state:")
print(pd.Series(celltype).value_counts().reindex(CELLTYPE_STATES).to_string())

## 3. Core metric functions (pseudobulk W and matched pseudobulk expression)

For every state (a set of cells) we aggregate raw UMIs across cells, then per multi-site gene:

- UTR length `W = sum_k rank_k * pi_k`, where `pi_k` are the within-gene PAS proportions of
  the aggregated counts (this is the pseudobulk analogue of the paper's per-cell W; using
  proportions removes the gene-count effect entirely).
- Expression `log1p(CPM)` of the gene-total UMIs (same molecules), the matched abundance signal.

A `(gene, state)` value is set to NaN when the gene has `< MIN_READS` aggregated UMIs there, so a
state is never scored on a gene it barely detects. Correlations between states are then computed
pairwise-complete (each state pair uses genes covered in both), which lets the small terminal
Neuron group (214 cells) join without decimating the gene set.


In [ ]:
def aggregate_site_sums(X, masks: list[np.ndarray]) -> np.ndarray:
    """Sum UMIs across cells within each state -> (n_states, n_sites)."""
    return np.vstack([np.asarray(X[m].sum(axis=0)).ravel() for m in masks])


def profiles_from_site_sums(
    site_sums: np.ndarray, min_reads: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Per-gene W and log-CPM expression per state from aggregated site sums.

    Returns (W, logcpm, gene_total) each shaped (n_genes, n_states); W/logcpm are NaN
    where the gene has < min_reads aggregated UMIs in that state.
    """
    n_states = site_sums.shape[0]
    libsize = site_sums.sum(axis=1)  # total UMIs per state
    W = np.full((len(genes), n_states), np.nan)
    logcpm = np.full((len(genes), n_states), np.nan)
    gene_total = np.zeros((len(genes), n_states))
    for gi, g in enumerate(genes):
        idx = g2idx[g]
        sub = site_sums[:, idx]  # (n_states, K)
        gsum = sub.sum(axis=1)  # (n_states,)
        gene_total[gi] = gsum
        ok = gsum >= min_reads
        if ok.any():
            props = sub[ok] / gsum[ok, None]
            W[gi, ok] = props @ gene_ranks[g]
            logcpm[gi, ok] = np.log1p(gsum[ok] / libsize[ok] * 1e6)
    return W, logcpm, gene_total


def pairwise_corr(P: np.ndarray, method: str = "pearson") -> np.ndarray:
    """State x state correlation over gene features, ignoring NaNs pairwise."""
    ns = P.shape[1]
    C = np.eye(ns)
    for i in range(ns):
        for j in range(i + 1, ns):
            m = np.isfinite(P[:, i]) & np.isfinite(P[:, j])
            a, b = P[m, i], P[m, j]
            if m.sum() < 3 or a.std() == 0 or b.std() == 0:
                c = np.nan
            elif method == "spearman":
                c = np.corrcoef(rankdata(a), rankdata(b))[0, 1]
            else:
                c = np.corrcoef(a, b)[0, 1]
            C[i, j] = C[j, i] = c
    return C


def top_variance_features(P: np.ndarray, covered: np.ndarray, top_n: int) -> np.ndarray:
    """Indices of the top-N genes by cross-state variance, among `covered` genes only."""
    v = np.full(P.shape[0], np.nan)
    v[covered] = np.nanvar(P[covered], axis=1)
    order = np.argsort(v)[::-1]
    order = order[np.isfinite(v[order])]  # never return uncovered genes
    return order[:top_n]


def mantel(D1: np.ndarray, D2: np.ndarray, n_perm: int, rng) -> tuple[float, float]:
    """Mantel correlation + permutation p between two square distance matrices."""
    iu = np.triu_indices_from(D1, k=1)
    a, b = D1[iu], D2[iu]
    keep = np.isfinite(a) & np.isfinite(b)
    a, b = a[keep], b[keep]
    r_obs = pearsonr(a, b)[0]
    n = D1.shape[0]
    perms = []
    import itertools

    if n <= 7:  # exact: enumerate all row/col permutations
        all_perms = list(itertools.permutations(range(n)))
    else:
        all_perms = None
    count = total = 0
    for _ in range(n_perm):
        p = (
            np.array(all_perms[RNG.integers(len(all_perms))])
            if all_perms is not None
            else rng.permutation(n)
        )
        D2p = D2[np.ix_(p, p)]
        bp = D2p[iu][keep]
        rp = pearsonr(a, bp)[0]
        total += 1
        if abs(rp) >= abs(r_obs):
            count += 1
    return r_obs, (count + 1) / (total + 1)


def draw_corr_dendro(fig, gs_cell, C, labels, title, cmap, vmin, vmax, *, annotate=True):
    """Dendrogram (top) + independently clustered correlation heatmap for one modality.

    Each matrix is clustered on its own (average linkage on 1 - corr), so the two
    modalities can group the cell states differently (the comparison of interest).
    Tick labels are coloured with the paper Fig-1 cell-type palette.
    """
    inner = gridspec.GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs_cell, height_ratios=[0.22, 1.0], hspace=0.02
    )
    Cfill = np.array(C, dtype=float)  # NaN-safe linkage (sparse Neuron pairs)
    if not np.isfinite(Cfill).all():
        Cfill[~np.isfinite(Cfill)] = np.nanmin(Cfill[np.isfinite(Cfill)])
    D = 1 - Cfill
    np.fill_diagonal(D, 0.0)
    Z = linkage(squareform(D, checks=False), method="average")
    order = leaves_list(Z)
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("#dddddd")

    ax_d = fig.add_subplot(inner[0])
    dendrogram(Z, ax=ax_d, color_threshold=0, above_threshold_color="#555555", no_labels=True)
    ax_d.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    for s in ax_d.spines.values():
        s.set_visible(False)
    ax_d.set_xticks([])
    ax_d.set_yticks([])

    ax_h = fig.add_subplot(inner[1])
    Cr = C[np.ix_(order, order)]
    lab = [labels[i] for i in order]
    im = ax_h.imshow(Cr, cmap=cm, vmin=vmin, vmax=vmax, aspect="equal")
    ax_h.set_xticks(range(len(lab)))
    ax_h.set_yticks(range(len(lab)))
    ax_h.set_xticklabels(lab, rotation=90, fontsize=S.FONT_TICK)
    ax_h.set_yticklabels(lab, fontsize=S.FONT_TICK)
    for k, t in enumerate(ax_h.get_xticklabels()):
        t.set_color(S.paper_palette_celltypes.get(lab[k], "#333333"))
    for k, t in enumerate(ax_h.get_yticklabels()):
        t.set_color(S.paper_palette_celltypes.get(lab[k], "#333333"))
    if annotate:
        mid = (vmin + vmax) / 2
        for ii in range(len(lab)):
            for jj in range(len(lab)):
                if np.isfinite(Cr[ii, jj]):
                    ax_h.text(
                        jj,
                        ii,
                        f"{Cr[ii, jj]:.2f}",
                        ha="center",
                        va="center",
                        fontsize=4.0,
                        color="white" if Cr[ii, jj] < mid else "black",
                    )
    return im


def dendro_corr_figure(Cl, Cr_, ltitle, rtitle, corr_name, suptitle, save_name):
    """Two clustered corr matrices (left | right) + dendrograms, one shared colourbar."""
    vmn = np.nanmin([np.nanmin(Cl), np.nanmin(Cr_)])
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(96) / 300))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.045], wspace=0.28)
    draw_corr_dendro(fig, gs[0], Cl, CELLTYPE_STATES, ltitle, CORR_CMAP, vmn, 1.0)
    im = draw_corr_dendro(fig, gs[1], Cr_, CELLTYPE_STATES, rtitle, CORR_CMAP, vmn, 1.0)
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{corr_name} r (cell states)", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    fig.suptitle(suptitle, fontsize=S.FONT_LABEL, y=1.02)
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


print("functions defined")


def heatmap_fixed(ax, C, labels, cmap, vmin, vmax, *, title=None, show_y=True):
    """Correlation heatmap in the given (differentiation-lineage) order - no clustering reorder."""
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("#dddddd")
    im = ax.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="equal")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=S.FONT_TICK)
    ax.set_yticklabels(labels if show_y else [], fontsize=S.FONT_TICK)
    for k, t in enumerate(ax.get_xticklabels()):
        t.set_color(S.paper_palette_celltypes.get(labels[k], "#333333"))
    if show_y:
        for k, t in enumerate(ax.get_yticklabels()):
            t.set_color(S.paper_palette_celltypes.get(labels[k], "#333333"))
    mid = (vmin + vmax) / 2
    for ii in range(len(labels)):
        for jj in range(len(labels)):
            if np.isfinite(C[ii, jj]):
                ax.text(
                    jj,
                    ii,
                    f"{C[ii, jj]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=4.0,
                    color="white" if C[ii, jj] < mid else "black",
                )
    if title:
        ax.set_title(title, fontsize=S.FONT_LABEL, pad=3)
    return im


def fixed_corr_figure(Cl, Cr_, ltitle, rtitle, corr_name, suptitle, save_name):
    """Two lineage-ordered corr matrices (left | right) + shared colourbar (no dendrogram)."""
    vmn = np.nanmin([np.nanmin(Cl), np.nanmin(Cr_)])
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(88) / 300))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.045], wspace=0.30)
    ax0 = fig.add_subplot(gs[0])
    heatmap_fixed(ax0, Cl, CELLTYPE_STATES, CORR_CMAP, vmn, 1.0, title=ltitle, show_y=True)
    ax1 = fig.add_subplot(gs[1])
    im = heatmap_fixed(ax1, Cr_, CELLTYPE_STATES, CORR_CMAP, vmn, 1.0, title=rtitle, show_y=False)
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{corr_name} r (cell states)", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    fig.suptitle(suptitle, fontsize=S.FONT_LABEL, y=1.0)
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)

## 4. Cell-type-collapsed comparison (Kremer Fig. 2a analogue)

State profiles for the 7 cell types. The heatmaps use each modality's own top-variance genes
(Kremer used expressed genes for RNA and high-coverage VMRs for methylation - i.e. the variable
features of each modality); the Mantel test and per-gene analyses below use the full common
gene set (unbiased). Pearson is primary (matching Kremer); Spearman is reported as robustness.


In [ ]:
ct_masks = [celltype == s for s in CELLTYPE_STATES]
ct_sums = aggregate_site_sums(Xutr, ct_masks)
W_ct, EXP_ct, tot_ct = profiles_from_site_sums(ct_sums, MIN_READS)

covered_core = np.isfinite(W_ct[:, :6]).all(axis=1)  # covered in all 6 core states
n_neuron_cov = int(np.isfinite(W_ct[covered_core, 6]).sum())
print(f"genes covered in all 6 core states: {covered_core.sum()}")
print(f"  ...of those, also covered in Neurons: {n_neuron_cov}")

# feature sets (per modality, top-variance among core-covered genes)
featW = top_variance_features(W_ct, covered_core, TOP_N_FEATURES)
featE = top_variance_features(EXP_ct, covered_core, TOP_N_FEATURES)

Cw = pairwise_corr(W_ct[featW])
Ce = pairwise_corr(EXP_ct[featE])
Cw_sp = pairwise_corr(W_ct[featW], "spearman")
Ce_sp = pairwise_corr(EXP_ct[featE], "spearman")

# Mantel on the common gene set (same genes, both modalities), core-6 only (Neurons NaN-heavy)
common = covered_core
Dw = 1 - pairwise_corr(W_ct[common][:, :6])
De = 1 - pairwise_corr(EXP_ct[common][:, :6])
mantel_r, mantel_p = mantel(De, Dw, N_PERM, RNG)
print(
    f"\nMantel (expression vs UTRome state geometry, 6 core types): "
    f"r = {mantel_r:.3f}, p = {mantel_p:.4f}  (n common genes = {common.sum()})"
)

# dendrogram agreement (cophenetic-distance correlation)
Lw = linkage(squareform(Dw, checks=False), method="average")
Le = linkage(squareform(De, checks=False), method="average")
coph_r = pearsonr(cophenet(Le), cophenet(Lw))[0]
print(f"cophenetic-distance correlation between the two dendrograms: {coph_r:.3f}")

### Panel: cell-type correlation matrices (differentiation-lineage order; Pearson and Spearman as separate files)

Cell types are shown in fixed differentiation order (qNSC1 -> qNSC2 -> aNSC1 -> aNSC2 -> TAP -> NB, then
terminal Neurons), not reordered by clustering, so the two modalities are directly comparable
cell-for-cell. Tick labels are coloured by the paper Fig-1 palette. Pearson and Spearman are written as
separate files (`..._corrmat_celltypes` and `..._corrmat_celltypes__spearman`; near-identical, pick
one). A dendrogram would re-impose clustering order, so it is omitted here; the clustering-topology
divergence between the two modalities is instead reported numerically (cophenetic r, stdout above).


In [ ]:
fixed_corr_figure(
    Ce,
    Cw,
    "Transcriptome (gene expression)",
    "3'UTRome (UTR-length index W)",
    "Pearson",
    f"Cell states: expression vs 3'UTR-length (Pearson)  (Mantel r={mantel_r:.2f}, "
    f"p={mantel_p:.3f}; clustering coph. r={coph_r:.2f})",
    "utrome_vs_transcriptome_corrmat_celltypes",
)

fixed_corr_figure(
    Ce_sp,
    Cw_sp,
    "Transcriptome (gene expression)",
    "3'UTRome (UTR-length index W)",
    "Spearman",
    "Cell states: expression vs 3'UTR-length (Spearman)",
    "utrome_vs_transcriptome_corrmat_celltypes__spearman",
)

## 5. Finely binned over pseudotime (equal-frequency bins)

Now resolve the trajectory at a finer grain than cell types: `N_BINS` equal-frequency
(population-quantile) bins over the 6 core lineage cells with valid pseudotime. Equal-frequency
binning guarantees each bin has enough cells for a stable pseudobulk estimate; with ~13.5k cells
and 20 bins that is ~675 cells/bin (NB alone spans many bins, so within-neuroblast changes are
resolved rather than collapsed).


In [ ]:
core = np.isin(celltype, CORE_LINEAGE) & np.isfinite(pt)
pt_c = pt[core]
Xc = Xutr[core]

edges = np.quantile(pt_c, np.linspace(0, 1, N_BINS + 1))
edges[0] -= 1e-9
bin_of = np.clip(np.digitize(pt_c, edges) - 1, 0, N_BINS - 1)
bin_centers = 0.5 * (edges[:-1] + edges[1:])
bin_masks = [bin_of == b for b in range(N_BINS)]
print("bin sizes:", np.bincount(bin_of, minlength=N_BINS))

bin_sums = aggregate_site_sums(Xc, bin_masks)
W_bin, EXP_bin, tot_bin = profiles_from_site_sums(bin_sums, MIN_READS_BIN)

covered_bin = np.isfinite(W_bin).all(axis=1)  # complete-case across all bins
print(f"genes covered >= {MIN_READS_BIN} UMIs in ALL {N_BINS} bins: {covered_bin.sum()}")

featWb = top_variance_features(W_bin, covered_bin, TOP_N_FEATURES)
featEb = top_variance_features(EXP_bin, covered_bin, TOP_N_FEATURES)
Cwb = pairwise_corr(W_bin[featWb])
Ceb = pairwise_corr(EXP_bin[featEb])
Cwb_sp = pairwise_corr(W_bin[featWb], "spearman")
Ceb_sp = pairwise_corr(EXP_bin[featEb], "spearman")

# majority cell type per bin -> axis colour strips (paper Fig-1 palette)
from matplotlib.colors import to_rgb

ct_core = celltype[core]
bin_majority = np.array(
    [pd.Series(ct_core[bin_of == b]).value_counts().idxmax() for b in range(N_BINS)]
)
bin_rgb = np.array([to_rgb(S.paper_palette_celltypes.get(c, "#999999")) for c in bin_majority])
print("bin majority cell types:", list(bin_majority))

# Mantel on the common (complete-case) gene set
Dwb = 1 - pairwise_corr(W_bin[covered_bin])
Deb = 1 - pairwise_corr(EXP_bin[covered_bin])
mantel_r_b, mantel_p_b = mantel(Deb, Dwb, N_PERM, RNG)
print(
    f"Mantel over pseudotime bins: r = {mantel_r_b:.3f}, p = {mantel_p_b:.4f} "
    f"(n common genes = {covered_bin.sum()})"
)

### Panel: pseudotime-bin correlation matrices (Pearson + Spearman)

Bins are kept in pseudotime order (trajectory geometry). Each matrix is flanked by cell-type colour
strips on the x and y axes (Kremer Fig. 2a style): each bin is coloured by its majority cell type
using the paper Fig-1 palette, so the qNSC->NB progression is visible along the axes.


In [ ]:
from matplotlib.patches import Patch


def bins_heatmap_strips(fig, spec, C, cmap, vmin, vmax, *, title=None):
    """20x20 bin correlation heatmap with majority-cell-type colour strips on both axes."""
    inner = gridspec.GridSpecFromSubplotSpec(
        2,
        2,
        subplot_spec=spec,
        width_ratios=[0.06, 1],
        height_ratios=[0.06, 1],
        wspace=0.02,
        hspace=0.02,
    )
    fig.add_subplot(inner[0, 0]).axis("off")
    ax_cx = fig.add_subplot(inner[0, 1])  # top x-strip
    ax_cy = fig.add_subplot(inner[1, 0])  # left y-strip
    ax_h = fig.add_subplot(inner[1, 1])
    cm = plt.get_cmap(cmap).copy()
    cm.set_bad("#dddddd")
    im = ax_h.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.set_xlabel("pseudotime bin →", fontsize=S.FONT_TICK)
    ax_cx.imshow(bin_rgb.reshape(1, -1, 3), aspect="auto")
    ax_cy.imshow(bin_rgb.reshape(-1, 1, 3), aspect="auto")
    for a in (ax_cx, ax_cy):
        a.set_xticks([])
        a.set_yticks([])
        for s in a.spines.values():
            s.set_visible(False)
    if title:
        ax_cx.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    return im


def strips_corr_figure(Ctx, Cutr, corr_name, suptitle, save_name):
    """Two bin corr matrices (Transcriptome | UTRome) with axis cell-type strips + legend."""
    vmn = np.nanmin([np.nanmin(Ctx), np.nanmin(Cutr)])
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(108) / 300))
    # content block shifted up: bottom=0.18 gives ~19mm below for legend,
    # top=0.82 keeps content height ~69mm (same as original); suptitle sits in the ~19mm above
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.045], wspace=0.16, top=0.82, bottom=0.18)
    bins_heatmap_strips(fig, gs[0], Ctx, CORR_CMAP, vmn, 1.0, title="Transcriptome")
    im = bins_heatmap_strips(fig, gs[1], Cutr, CORR_CMAP, vmn, 1.0, title="3'UTRome (W)")
    cax = fig.add_subplot(gs[2])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{corr_name} r (bins)", fontsize=S.FONT_TICK)
    cax.tick_params(labelsize=S.FONT_TICK)
    seen = [c for c in CELLTYPE_STATES if c in set(bin_majority)]  # lineage order
    handles = [Patch(facecolor=S.paper_palette_celltypes[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(handles),
        frameon=False,
        fontsize=S.FONT_LEGEND,
        bbox_to_anchor=(0.5, 0.05),
        title="axis strip = bin majority cell type",
        title_fontsize=S.FONT_LEGEND_TITLE,
    )
    fig.suptitle(suptitle, fontsize=S.FONT_LABEL, y=0.95)
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


strips_corr_figure(
    Ceb,
    Cwb,
    "Pearson",
    f"Pseudotime bins ({N_BINS}): expression vs 3'UTR-length (Pearson)  "
    f"(Mantel r={mantel_r_b:.2f}, p={mantel_p_b:.3f})",
    "utrome_vs_transcriptome_corrmat_pseudotime_bins",
)

strips_corr_figure(
    Ceb_sp,
    Cwb_sp,
    "Spearman",
    f"Pseudotime bins ({N_BINS}): expression vs 3'UTR-length (Spearman)",
    "utrome_vs_transcriptome_corrmat_pseudotime_bins__spearman",
)

### Panel: Spearman pseudotime bins with independent colour scales (diagnostic)

Same Spearman bin matrices as directly above, but each modality is scaled to its own [min, max]
instead of a shared range, and each panel gets its own colourbar. The 3'UTRome spans a much narrower
correlation range than the transcriptome, so on the shared scale its internal structure looks flat;
rescaling each panel to its own min/max reveals the relative bin-to-bin contrast within the UTRome,
for visual comparison with the transcriptome's pattern. Trade-off: absolute r values are not
comparable across the two panels here (the colourbars span different ranges); use the shared-scale
version above when comparing magnitudes.


In [ ]:
def strips_corr_figure_indep(Ctx, Cutr, corr_name, suptitle, save_name):
    """Bin corr matrices with axis cell-type strips, but each modality on its own colour
    scale (vmin/vmax = that matrix\'s own min/max) + its own colourbar."""
    fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(108) / 300))
    gs = gridspec.GridSpec(
        1, 4, width_ratios=[1, 0.05, 1, 0.05], wspace=0.30, top=0.82, bottom=0.18
    )
    for hcol, ccol, C, ttl, short in [
        (0, 1, Ctx, "Transcriptome", "Tx"),
        (2, 3, Cutr, "3'UTRome (W)", "UTRome"),
    ]:
        im = bins_heatmap_strips(fig, gs[hcol], C, CORR_CMAP, np.nanmin(C), np.nanmax(C), title=ttl)
        cax = fig.add_subplot(gs[ccol])
        cb = fig.colorbar(im, cax=cax)
        cb.set_label(f"{corr_name} r ({short}, own scale)", fontsize=S.FONT_TICK)
        cax.tick_params(labelsize=S.FONT_TICK)
    seen = [c for c in CELLTYPE_STATES if c in set(bin_majority)]
    handles = [Patch(facecolor=S.paper_palette_celltypes[c], label=c) for c in seen]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(handles),
        frameon=False,
        fontsize=S.FONT_LEGEND,
        bbox_to_anchor=(0.5, 0.05),
        title="axis strip = bin majority cell type",
        title_fontsize=S.FONT_LEGEND_TITLE,
    )
    fig.suptitle(suptitle, fontsize=S.FONT_LABEL, y=0.95)
    S.save_and_show(fig, save_name, fig_dir=UNASSIGNED_DIR)


strips_corr_figure_indep(
    Ceb_sp,
    Cwb_sp,
    "Spearman",
    f"Pseudotime bins ({N_BINS}): expression vs 3'UTR-length (Spearman, INDEPENDENT colour scales)",
    "utrome_vs_transcriptome_corrmat_pseudotime_bins__spearman_freescale",
)

### Variant: quarter-height x full-width, square heatmaps (freescale)

Alternative layout of the freescale Spearman corrmat above, for figure remodelling:
full width x quarter height (174 x 58.75 mm), both heatmaps forced to a square
aspect ratio, each with its own colourbar placed immediately to the right of its
own heatmap (so the left colourbar no longer overlaps the right panel), the suptitle
removed (the description belongs in the figure legend), and the cell-type strip key
as a compact vertical legend on the right. Saved as
`..._spearman_freescale__quarter`.


In [ ]:
# --- Variant: quarter-height x full-width, square heatmaps (freescale) ---
# Alternative layout of the freescale corrmat for figure remodelling: full width x quarter
# height (S.FULL_W_IN x S.QUARTER_H_IN = 174 x 58.75 mm), both heatmaps forced to a square
# aspect, each with its own colourbar placed right next to its own heatmap (no overlap with
# the neighbouring panel), no suptitle (goes in the fig legend), and a compact vertical
# cell-type legend on the right. Reuses Ceb_sp / Cwb_sp / bin_rgb / bin_majority from above.
from matplotlib.patches import Patch

_FIGW_MM, _FIGH_MM = S.FULL_W_MM, S.QUARTER_H_MM  # 174 x 58.75 mm


def _square_corr_block(
    fig,
    xleft,
    C,
    vmin,
    vmax,
    title,
    *,
    side=45.0,
    ybot=7.0,
    strip=1.8,
    sgap=0.3,
    tstrip=2.0,
    tgap=0.6,
    cgap=1.5,
    cw=1.8,
    cbar_frac=1.0,
):
    """Place a square (side x side mm) bin-corr heatmap + cell-type strips + its own
    colourbar via explicit mm->figure-fraction axes, so the heatmap is exactly square and
    the strips stay aligned. `xleft` = left edge of the y-strip, in mm."""
    fx = lambda v: v / _FIGW_MM
    fy = lambda v: v / _FIGH_MM
    hx = xleft + strip + sgap  # heatmap left edge (mm)
    ax_h = fig.add_axes([fx(hx), fy(ybot), fx(side), fy(side)])  # square in physical mm
    cm = plt.get_cmap(CORR_CMAP).copy()
    cm.set_bad("#dddddd")
    im = ax_h.imshow(C, cmap=cm, vmin=vmin, vmax=vmax, aspect="auto", origin="upper")
    ax_h.set_xticks([])
    ax_h.set_yticks([])
    ax_h.set_xlabel("pseudotime bin →", fontsize=S.FONT_TICK, labelpad=2)
    ax_cy = fig.add_axes([fx(xleft), fy(ybot), fx(strip), fy(side)])  # left y-strip
    ax_cy.imshow(bin_rgb.reshape(-1, 1, 3), aspect="auto")
    ax_cx = fig.add_axes([fx(hx), fy(ybot + side + tgap), fx(side), fy(tstrip)])  # top x-strip
    ax_cx.imshow(bin_rgb.reshape(1, -1, 3), aspect="auto")
    for a in (ax_cy, ax_cx):
        a.set_xticks([])
        a.set_yticks([])
        for s in a.spines.values():
            s.set_visible(False)
    if title:
        ax_cx.set_title(title, fontsize=S.FONT_LABEL, pad=2)
    cax = fig.add_axes(
        [fx(hx + side + cgap), fy(ybot), fx(cw), fy(side * cbar_frac)]
    )  # own colourbar
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Spearman r", fontsize=S.FONT_TICK, labelpad=1)
    cax.tick_params(labelsize=S.FONT_TICK, pad=1)
    return im


fig = plt.figure(figsize=(S.FULL_W_IN, S.QUARTER_H_IN))
_square_corr_block(fig, 4.0, Ceb_sp, np.nanmin(Ceb_sp), np.nanmax(Ceb_sp), "Transcriptome")
_square_corr_block(fig, 75.0, Cwb_sp, np.nanmin(Cwb_sp), np.nanmax(Cwb_sp), "3'UTRome (W)")

_seen = [c for c in CELLTYPE_STATES if c in set(bin_majority)]
_handles = [Patch(facecolor=S.paper_palette_celltypes[c], label=c) for c in _seen]
fig.legend(
    handles=_handles,
    loc="center left",
    bbox_to_anchor=(0.82, 0.5),
    ncol=1,
    frameon=False,
    fontsize=S.FONT_LEGEND,
    handlelength=1.1,
    handleheight=1.0,
    labelspacing=0.35,
    title="bin majority\ncell type",
    title_fontsize=S.FONT_LEGEND_TITLE,
)

S.save_and_show(
    fig,
    "utrome_vs_transcriptome_corrmat_pseudotime_bins__spearman_freescale__quarter",
    fig_dir=UNASSIGNED_DIR,
)

In [ ]:
# --- Short-colourbar variant: same quarter panel, correlation colourbar much shorter
# (cbar_frac=0.4) + bottom-aligned so it lands in the lower-right, outside the heatmaps. ---
fig = plt.figure(figsize=(S.FULL_W_IN, S.QUARTER_H_IN))
_square_corr_block(
    fig,
    4.0,
    Ceb_sp,
    np.nanmin(Ceb_sp),
    np.nanmax(Ceb_sp),
    "Transcriptome",
    cbar_frac=0.4,
)
_square_corr_block(
    fig,
    75.0,
    Cwb_sp,
    np.nanmin(Cwb_sp),
    np.nanmax(Cwb_sp),
    "3'UTRome (W)",
    cbar_frac=0.4,
)
_seen = [c for c in CELLTYPE_STATES if c in set(bin_majority)]
_handles = [Patch(facecolor=S.paper_palette_celltypes[c], label=c) for c in _seen]
fig.legend(
    handles=_handles,
    loc="center left",
    bbox_to_anchor=(0.82, 0.5),
    ncol=1,
    frameon=False,
    fontsize=S.FONT_LEGEND,
    handlelength=1.1,
    handleheight=1.0,
    labelspacing=0.35,
    title="bin majority\ncell type",
    title_fontsize=S.FONT_LEGEND_TITLE,
)
S.save_and_show(
    fig,
    "utrome_vs_transcriptome_corrmat_pseudotime_bins__spearman_freescale__quarter_shortcbar",
    fig_dir=UNASSIGNED_DIR,
)

### Panel: correlation-vs-bin-distance decay (rate/geometry of change per modality)

If both modalities changed at the same rate along pseudotime the two decay curves would overlap.
A shallower UTRome curve = UTR length keeps changing across the whole trajectory; a curve that
turns back up at large lag = the trajectory folds back (non-monotonic) in that modality.


In [ ]:
def decay_curve(C: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n = C.shape[0]
    lags = range(1, n)
    means = [np.nanmean([C[i, i + d] for i in range(n - d)]) for d in lags]
    return np.array(list(lags)), np.array(means)


lags, dec_e = decay_curve(Ceb)
_, dec_w = decay_curve(Cwb)

fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(62) / 300))
ax.plot(lags, dec_e, "-o", color=COL_TX, ms=3, lw=1.3, label="Transcriptome")
ax.plot(lags, dec_w, "-o", color=COL_UTR, ms=3, lw=1.3, label="3'UTRome (W)")
ax.set_xlabel("pseudotime-bin separation (lag)", fontsize=S.FONT_LABEL)
ax.set_ylabel("mean Pearson r", fontsize=S.FONT_LABEL)
ax.set_title("How fast each modality changes along pseudotime", fontsize=S.FONT_LABEL)
ax.legend(frameon=False, fontsize=S.FONT_LEGEND)
ax.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "utrome_transcriptome_correlation_decay_pseudotime", fig_dir=UNASSIGNED_DIR)

## 6. Per-gene decoupling of UTR length and expression

For each well-covered gene we take its expression
trajectory (log-CPM over the pseudotime bins) and its UTR-length trajectory (W over the
same bins) and correlate them (Spearman across bins). If UTR length were merely a readout of
expression these would cluster near +/-1; a distribution centred near 0 means the two layers move
independently. We also measure each gene's dynamic range in both layers to find genes that
shift their 3'UTR strongly while their expression stays flat, i.e. length changes not visible in the
transcriptome. APA-dynamic genes (Fig-1 LRT, FDR<5%) are highlighted.

How to read the two panels below. Left (distribution): for every well-covered gene we take its
expression trajectory (log-CPM across the 20 pseudotime bins) and its 3'UTR-length trajectory (W across
the same bins) and compute one Spearman correlation between them. Each gene is one value on the x-axis:
+1 = the 3'UTR lengthens exactly as expression rises (fully coupled), -1 = lengthens as expression
falls, 0 = the two layers move independently. A histogram piled at +/-1 would mean "UTR length is just
a readout of expression"; a histogram centred on 0 (what we see) means they are largely independent.
Purple = APA-dynamic genes (Fig-1 LRT). Right (scatter): each gene is placed by how much it changes in
each layer over the trajectory: x = expression dynamic range (max-min log-CPM), y = 3'UTR-length range
(max-min W); dotted lines are the medians. Genes in the upper-left remodel their 3'UTR a lot while
expression barely moves ("UTR-only"); the lower-right is the opposite. Labels mark the most extreme
UTR-only APA-dynamic genes.


In [ ]:
rank = pd.read_csv(RANKING_CSV)[["gene_id", "gene_name", "padj", "delta_psi", "rho_W", "direction"]]

rows = []
gi_of = {g: i for i, g in enumerate(genes)}
for g in np.array(genes)[covered_bin]:
    i = gi_of[g]
    e = EXP_bin[i]
    w = W_bin[i]
    rho = spearmanr(e, w).correlation if (np.std(e) > 0 and np.std(w) > 0) else np.nan
    rows.append(
        {
            "gene_id": g,
            "gene_name": gene_sym[g],
            "rho_expr_W": rho,
            "expr_range": float(np.nanmax(e) - np.nanmin(e)),
            "W_range": float(np.nanmax(w) - np.nanmin(w)),
        }
    )
dec = pd.DataFrame(rows).merge(rank, on="gene_id", how="left", suffixes=("", "_rk"))
dec["sig_apa"] = dec["padj"] < 0.05

med_abs = dec["rho_expr_W"].abs().median()
frac_lo = (dec["rho_expr_W"].abs() < 0.3).mean()
# median-split quadrants
me, mw = dec["expr_range"].median(), dec["W_range"].median()
utr_only = dec[(dec["expr_range"] < me) & (dec["W_range"] >= mw)]
expr_only = dec[(dec["expr_range"] >= me) & (dec["W_range"] < mw)]
sig = dec[dec["sig_apa"]]
print(f"n genes: {len(dec)} | APA-dynamic (FDR<5%): {int(dec['sig_apa'].sum())}")
print(f"median |Spearman(expr traj, W traj)| = {med_abs:.3f}; fraction |rho|<0.3 = {frac_lo:.1%}")
print(
    f"'UTR-only' genes (low expr range, high W range): {len(utr_only)} "
    f"({int(utr_only['sig_apa'].sum())} APA-dynamic)"
)
print(f"among APA-dynamic genes, median |Spearman| = {sig['rho_expr_W'].abs().median():.3f}")
dec.to_csv(RESULTS_DIR / "expression_vs_utrlength_decoupling_per_gene.csv", index=False)
print("saved:", RESULTS_DIR / "expression_vs_utrlength_decoupling_per_gene.csv")

### Panel: per-gene decoupling (distribution + dynamic-range scatter)


In [ ]:
try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False

fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(70) / 300))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.25], wspace=0.28)

# (a) distribution of per-gene expr-vs-W trajectory correlation
axh = fig.add_subplot(gs[0])
axh.hist(
    dec["rho_expr_W"].dropna(),
    bins=40,
    color="#BBBBBB",
    edgecolor="white",
    lw=0.3,
    label="all covered genes",
)
axh.hist(
    sig["rho_expr_W"].dropna(),
    bins=40,
    color=COL_UTR,
    alpha=0.75,
    label="APA-dynamic (FDR<5%)",
)
axh.axvline(0, color="k", lw=0.6, ls="--")
axh.set_xlabel("Spearman(expression traj., UTR-length traj.)", fontsize=S.FONT_LABEL)
axh.set_ylabel("genes", fontsize=S.FONT_LABEL)
axh.set_title(
    f"Per-gene decoupling\nmedian |r|={med_abs:.2f}, |r|<0.3 for {frac_lo:.0%}",
    fontsize=S.FONT_LABEL,
)
axh.legend(frameon=False, fontsize=S.FONT_LEGEND)
axh.tick_params(labelsize=S.FONT_TICK)

# (b) dynamic-range scatter: UTR-length change vs expression change
axs = fig.add_subplot(gs[1])
ns_ = dec[~dec["sig_apa"]]
axs.scatter(ns_["expr_range"], ns_["W_range"], s=5, c="#CFCFCF", lw=0, label="not APA-dynamic")
axs.scatter(
    sig["expr_range"],
    sig["W_range"],
    s=8,
    c=COL_UTR,
    lw=0,
    alpha=0.8,
    label="APA-dynamic (FDR<5%)",
)
axs.axvline(me, color="k", lw=0.5, ls=":")
axs.axhline(mw, color="k", lw=0.5, ls=":")
axs.set_xlabel("expression dynamic range over pseudotime (log-CPM max-min)", fontsize=S.FONT_LABEL)
axs.set_ylabel("3'UTR-length range (W max-min)", fontsize=S.FONT_LABEL)
axs.set_title(f"UTR-only genes (upper-left): {len(utr_only)}", fontsize=S.FONT_LABEL)
axs.legend(frameon=False, fontsize=S.FONT_LEGEND, loc="upper right")
axs.tick_params(labelsize=S.FONT_TICK)
axs.set_xlim(0, float(np.percentile(dec["expr_range"], 98)))  # spread the bulk; rare outliers clip
# label the most extreme UTR-only APA-dynamic genes
lab_src = utr_only[utr_only["sig_apa"]].nlargest(8, "W_range")
texts = [
    axs.text(r.expr_range, r.W_range, r.gene_name, fontsize=4.6, style="italic")
    for r in lab_src.itertuples()
]
if HAS_AT and texts:
    adjust_text(texts, ax=axs, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4))
fig.tight_layout()
S.save_and_show(fig, "per_gene_expression_vs_utrlength_decoupling", fig_dir=UNASSIGNED_DIR)

## 7. Information in the 3'UTRome that is not in the transcriptome (direct tests)

If UTR length were a redundant readout of expression, a gene's expression trajectory would
predict its UTR-length trajectory; the per-gene decoupling (Section 6) argues against this.
Here we add three model-free tests:

1. Expression-invisible regulation: genes that are APA-dynamic (Fig-1 LRT, FDR<5%) yet show no
   expression trend along pseudotime: their regulation is undetectable in the transcriptome.
2. Direction (only partial coupling): among genes changing in both layers, 3'UTR
   lengthen/shorten is only weakly predictable from expression up/down: a modest proliferation-linked
   bias (expression-up leans toward shortening) with many exceptions, so the sign is far from
   determined by expression (Fisher test).
3. Change timing: for genes changing in both layers, the pseudotime at which the 3'UTR shift
   happens differs from the expression shift, so the 3'UTRome reports temporally distinct events.


In [ ]:
from scipy.stats import fisher_exact, wilcoxon

bin_idx = np.arange(N_BINS)
cov_genes = np.array(genes)[covered_bin]
expr_tr = np.array([spearmanr(EXP_bin[gi_of[g]], bin_idx).correlation for g in cov_genes])
W_tr = np.array([spearmanr(W_bin[gi_of[g]], bin_idx).correlation for g in cov_genes])
trend = pd.DataFrame(
    {
        "gene_id": cov_genes,
        "gene_name": [gene_sym[g] for g in cov_genes],
        "expr_trend": expr_tr,
        "W_trend": W_tr,
    }
)
trend = trend.merge(rank[["gene_id", "padj", "delta_psi", "rho_W"]], on="gene_id", how="left")
trend["sig_apa"] = trend["padj"] < 0.05

TAU = 0.5  # |Spearman trend| effect-size gate for "dynamic in a layer"
FLAT = 0.3  # |expr trend| below this = expression-flat

# (1) expression-invisible APA regulation
apa_dyn = trend["sig_apa"] & (trend["W_trend"].abs() >= TAU)
invisible = trend[apa_dyn & (trend["expr_trend"].abs() < FLAT)]
print(f"APA-dynamic genes with a clear UTR trend (|W trend|>={TAU}): {int(apa_dyn.sum())}")
print(
    f"  ...expression-flat (|expr trend|<{FLAT}): {len(invisible)} "
    f"({len(invisible) / max(int(apa_dyn.sum()), 1):.0%}) -> invisible to the transcriptome"
)

# (2) direction independence among genes dynamic in both layers
both = trend[(trend["expr_trend"].abs() >= TAU) & (trend["W_trend"].abs() >= TAU)].copy()
both["expr_dir"] = np.where(both["expr_trend"] > 0, "expr up", "expr down")
both["W_dir"] = np.where(both["W_trend"] > 0, "UTR lengthen", "UTR shorten")
ct2 = (
    pd.crosstab(both["expr_dir"], both["W_dir"])
    .reindex(index=["expr up", "expr down"], columns=["UTR lengthen", "UTR shorten"])
    .fillna(0)
    .astype(int)
)
OR, p_dir = fisher_exact(ct2.values)
print(f"\ngenes dynamic in BOTH layers (|trend|>={TAU}): {len(both)}")
print(ct2.to_string())
up_len = int(ct2.loc["expr up", "UTR lengthen"])
up_tot = int(ct2.loc["expr up"].sum())
print(
    f"direction coupling: Fisher OR={OR:.2f}, p={p_dir:.3f} "
    f"(modest expr-up<->UTR-shorten bias; {up_len}/{up_tot} expr-up genes still lengthen "
    f"=> sign far from determined by expression)"
)


# (3) change timing: pseudotime bin where half the total change has occurred
def change_midpoint(traj):
    d = np.abs(np.diff(traj))
    return int(np.searchsorted(np.cumsum(d) / d.sum(), 0.5)) if d.sum() > 0 else np.nan


both["mid_expr"] = [change_midpoint(EXP_bin[gi_of[g]]) for g in both["gene_id"]]
both["mid_W"] = [change_midpoint(W_bin[gi_of[g]]) for g in both["gene_id"]]
both["offset"] = both["mid_W"] - both["mid_expr"]
med_off = both["offset"].median()
try:
    w_p = wilcoxon(both["offset"], zero_method="wilcox").pvalue
except ValueError:
    w_p = np.nan
print(
    f"\nchange-timing offset (UTR - expression): median = {med_off:.1f} bins; "
    f"Wilcoxon p={w_p:.1e}; |offset|>=3 bins for {(both['offset'].abs() >= 3).mean():.0%} of genes"
)

### Panel: APA-dynamic but expression-flat (regulation invisible to the transcriptome)

How to read. Each gene is placed by its signed trajectory trend: x = Spearman of its expression vs
pseudotime, y = Spearman of its 3'UTR-length (W) vs pseudotime (both in [-1, 1]; sign = direction,
magnitude = how monotonic). The grey vertical band is "expression-flat" (|expr trend| < 0.3). Purple =
APA-dynamic (Fig-1 LRT, FDR<5%). The genes of interest are the purple points inside the grey band with
large |y|: their 3'UTR shifts strongly and monotonically along neurogenesis while their expression does
not, so this regulation is not visible in the transcriptome. Labelled genes are the strongest such cases.


In [ ]:
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(66) / 300))
ns_ = trend[~trend["sig_apa"]]
si_ = trend[trend["sig_apa"]]
ax.axvspan(-FLAT, FLAT, color="#000000", alpha=0.06, lw=0)  # expression-flat band
ax.scatter(ns_["expr_trend"], ns_["W_trend"], s=5, c="#CFCFCF", lw=0, label="not APA-dynamic")
ax.scatter(
    si_["expr_trend"],
    si_["W_trend"],
    s=8,
    c=COL_UTR,
    lw=0,
    alpha=0.8,
    label="APA-dynamic",
)
ax.axhline(0, color="k", lw=0.4)
ax.axvline(0, color="k", lw=0.4)
ax.set_xlabel("expression trend (Spearman vs pseudotime)", fontsize=S.FONT_LABEL)
ax.set_ylabel("3'UTR-length trend (Spearman W vs pseudotime)", fontsize=S.FONT_LABEL)
ax.set_title(
    f"APA-dynamic but expression-flat: {len(invisible)} genes\n(shaded = no expression trend)",
    fontsize=S.FONT_LABEL,
)
ax.legend(frameon=False, fontsize=S.FONT_LEGEND, loc="lower right")
ax.tick_params(labelsize=S.FONT_TICK)
lab_src = invisible.loc[invisible["W_trend"].abs().sort_values(ascending=False).index].head(8)
texts = [
    ax.text(r.expr_trend, r.W_trend, r.gene_name, fontsize=4.6, style="italic")
    for r in lab_src.itertuples()
]
if HAS_AT and texts:
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4))
fig.tight_layout()
S.save_and_show(fig, "apa_dynamic_but_expression_flat", fig_dir=UNASSIGNED_DIR)

### Panel: direction independence (UTR lengthen/shorten vs expression up/down)

How to read. Restricted to genes that clearly change in both layers (|trend| >= 0.5 each). The left
pair of bars are genes whose expression rises along pseudotime, split into how many lengthen (red) vs
shorten (blue) their 3'UTR; the right pair are genes whose expression falls. If UTR direction were fully
set by expression direction the two groups would be clean opposites; instead both contain many of each.
The Fisher odds ratio and p quantify the (weak) association: OR<1, p~0.02 = a modest bias
(expression-up leans toward shortening, the known proliferation signature) with many exceptions, so
3'UTR direction is only partly predictable from expression.


In [ ]:
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(62) / 300))
groups = ["expr up", "expr down"]
x = np.arange(2)
bw = 0.38
ax.bar(
    x - bw / 2,
    [ct2.loc[g, "UTR lengthen"] for g in groups],
    bw,
    color=S.COL_LONG,
    label="UTR lengthen",
)
ax.bar(
    x + bw / 2,
    [ct2.loc[g, "UTR shorten"] for g in groups],
    bw,
    color=S.COL_SHORT,
    label="UTR shorten",
)
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=S.FONT_TICK)
ax.set_ylabel("genes (dynamic in both layers)", fontsize=S.FONT_LABEL)
ax.set_title(
    f"3'UTR direction only weakly coupled to expression\n"
    f"Fisher OR={OR:.2f}, p={p_dir:.2f} (many exceptions)",
    fontsize=S.FONT_LABEL,
)
ax.legend(frameon=False, fontsize=S.FONT_LEGEND)
ax.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "utrome_transcriptome_direction_independence", fig_dir=UNASSIGNED_DIR)

### Panel: change timing (when along pseudotime each layer changes)

How to read. For each gene changing in both layers we find its "change midpoint", the pseudotime bin
by which half of that gene's total change (in expression, or in W) has accumulated. Left: one point per
gene, x = expression midpoint, y = 3'UTR midpoint; points on the dashed diagonal change at the same time,
points off the diagonal change at different pseudotimes. Right: histogram of the per-gene offset
(3'UTR midpoint - expression midpoint); mass to the right of 0 = the 3'UTR shift tends to happen later
than the expression shift. The spread (|offset| >= 3 bins for ~half the genes) shows that the 3'UTRome
reports temporally distinct events.


In [ ]:
rng_j = np.random.default_rng(0)
fig, (axs, axh) = plt.subplots(
    1,
    2,
    figsize=(S.FULL_W_PX / 300, S.mm_to_px(66) / 300),
    gridspec_kw={"width_ratios": [1.15, 1]},
)
jx = both["mid_expr"] + rng_j.normal(0, 0.15, len(both))
jy = both["mid_W"] + rng_j.normal(0, 0.15, len(both))
axs.plot([0, N_BINS - 1], [0, N_BINS - 1], color="gray", lw=0.7, ls="--")
axs.scatter(jx, jy, s=6, c=COL_UTR, alpha=0.5, lw=0)
axs.set_xlabel("expression change midpoint (bin)", fontsize=S.FONT_LABEL)
axs.set_ylabel("3'UTR change midpoint (bin)", fontsize=S.FONT_LABEL)
axs.set_title("When each layer changes\n(off-diagonal = different timing)", fontsize=S.FONT_LABEL)
axs.tick_params(labelsize=S.FONT_TICK)
axh.hist(both["offset"], bins=range(-N_BINS, N_BINS + 2), color=COL_UTR, alpha=0.8)
axh.axvline(0, color="k", lw=0.7, ls="--")
axh.axvline(med_off, color="#C1121F", lw=1.3, label=f"median = {med_off:.0f}")
axh.set_xlabel("timing offset (UTR - expression, bins)", fontsize=S.FONT_LABEL)
axh.set_ylabel("genes", fontsize=S.FONT_LABEL)
axh.set_title("3'UTR change is later than\nexpression change", fontsize=S.FONT_LABEL)
axh.legend(frameon=False, fontsize=S.FONT_LEGEND)
axh.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "utrome_transcriptome_change_timing", fig_dir=UNASSIGNED_DIR)

### Panel: change timing for confident (LRT + permutation) APA-dynamic genes

The panel above keeps genes with a strong monotonic trend in both layers (|trend| >= 0.5), with no
significance filter. This focused version restricts to the genes we are most confident remodel their
3'UTR along neurogenesis: those significant by both the Fig-1 chi-squared LRT and the
within-sample permutation null (BH FDR<5% in each; the permutation null is the paper's primary
calibration). The "changes in both layers" gate is kept because the panel compares when expression
changes vs when the 3'UTR changes, so both trajectories need a defined change point.


In [ ]:
# Confident set: significant by both the chi-squared LRT and the within-sample permutation null
# (BH<0.05 in each; permutation = the paper's primary calibration), intersected with genes changing
# in both layers (needed to define an expression change-timing).
PERM_CSV = (
    UTR_BASE
    / "DirMult_optimized"
    / "permutation_batch_number"
    / "dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_cov-batch_number.csv"
)
perm = (
    pd.read_csv(PERM_CSV)[["utr_name", "padj"]]
    .drop_duplicates("utr_name")
    .rename(columns={"utr_name": "gene_id", "padj": "padj_perm"})
)
both = both.merge(perm, on="gene_id", how="left")
both["sig_confident"] = both["sig_apa"] & (both["padj_perm"] < 0.05)
bsig = both[both["sig_confident"]].copy()
med_off_s = bsig["offset"].median()
try:
    w_p_s = wilcoxon(bsig["offset"], zero_method="wilcox").pvalue
except ValueError:
    w_p_s = np.nan
print(
    f"confident (LRT + permutation) timing subset: {len(bsig)} genes of {len(both)} dynamic-in-both "
    f"({int(both['sig_apa'].sum())} LRT-sig; {len(bsig)} also permutation-sig); "
    f"median offset = {med_off_s:.1f} bins, Wilcoxon p={w_p_s:.1e}; "
    f"|offset|>=3 bins for {(bsig['offset'].abs() >= 3).mean():.0%}"
)

rng_j = np.random.default_rng(0)
fig, (axs, axh) = plt.subplots(
    1,
    2,
    figsize=(S.FULL_W_PX / 300, S.mm_to_px(66) / 300),
    gridspec_kw={"width_ratios": [1.15, 1]},
)
jx = bsig["mid_expr"] + rng_j.normal(0, 0.15, len(bsig))
jy = bsig["mid_W"] + rng_j.normal(0, 0.15, len(bsig))
axs.plot([0, N_BINS - 1], [0, N_BINS - 1], color="gray", lw=0.7, ls="--")
axs.scatter(jx, jy, s=7, c=COL_UTR, alpha=0.6, lw=0)
axs.set_xlabel("expression change midpoint (bin)", fontsize=S.FONT_LABEL)
axs.set_ylabel("3'UTR change midpoint (bin)", fontsize=S.FONT_LABEL)
axs.set_title(
    "When each layer changes - confident APA genes\n(off-diagonal = different timing)",
    fontsize=S.FONT_LABEL,
)
axs.tick_params(labelsize=S.FONT_TICK)
axh.hist(bsig["offset"], bins=range(-N_BINS, N_BINS + 2), color=COL_UTR, alpha=0.85)
axh.axvline(0, color="k", lw=0.7, ls="--")
axh.axvline(med_off_s, color="#C1121F", lw=1.3, label=f"median = {med_off_s:.0f}")
axh.set_xlabel("timing offset (UTR - expression, bins)", fontsize=S.FONT_LABEL)
axh.set_ylabel("genes", fontsize=S.FONT_LABEL)
axh.set_title(f"LRT + permutation significant (n={len(bsig)})", fontsize=S.FONT_LABEL)
axh.legend(frameon=False, fontsize=S.FONT_LEGEND)
axh.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "utrome_transcriptome_change_timing__confident", fig_dir=UNASSIGNED_DIR)

## 8. Cell-state similarity: the 3'UTR-length axis is not redundant with expression

Kremer et al. reported a specific dissociation (vSVZ astrocytes vs qNSC2 look alike in the
transcriptome but differ sharply in the methylome). We test the analogous question here: for every
cell-state pair, whether its 3'UTR-length similarity is predictable from its expression similarity. The
relation is not a straight line. Consistent with the decay curves (Section 5), most pairs sit above
y=x: across cell states 3'UTR length is the more conserved axis (expression changes more), so
the two similarity measures are non-redundant. The pairs closest to the diagonal are at the
differentiation front (TAP<->NB, aNSC<->TAP), where the 3'UTR layer contributes separation
comparable to expression.


In [ ]:
pairs = []
for i in range(len(CELLTYPE_STATES)):
    for j in range(i + 1, len(CELLTYPE_STATES)):
        if np.isfinite(Ce[i, j]) and np.isfinite(Cw[i, j]):
            pairs.append((CELLTYPE_STATES[i], CELLTYPE_STATES[j], Ce[i, j], Cw[i, j]))
pdf = pd.DataFrame(pairs, columns=["s1", "s2", "expr_r", "W_r"])
pdf["gap"] = pdf["expr_r"] - pdf["W_r"]  # >0: expression more similar than UTR length
pdf = pdf.sort_values("gap", ascending=False)
frac_above = (pdf["gap"] < 0).mean()  # UTR more similar than expression
print(f"state pairs where UTR length is MORE conserved than expression: {frac_above:.0%}")
print("pairs closest to the diagonal / most UTR-separated (top gap = expr more similar):")
print(pdf.head(6).to_string(index=False))

fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, S.mm_to_px(70) / 300))
lo = min(pdf.expr_r.min(), pdf.W_r.min()) - 0.02
ax.plot([lo, 1], [lo, 1], color="gray", lw=0.7, ls="--")
sc = ax.scatter(
    pdf.expr_r,
    pdf.W_r,
    s=14,
    c=pdf.gap,
    cmap="coolwarm",
    vmin=-abs(pdf.gap).max(),
    vmax=abs(pdf.gap).max(),
    lw=0.2,
    edgecolor="k",
)
texts = []
for r in pdf.head(4).itertuples():
    texts.append(ax.text(r.expr_r, r.W_r, f"{r.s1}-{r.s2}", fontsize=4.6))
if HAS_AT and texts:
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4))
ax.set_xlabel("expression similarity (Pearson r)", fontsize=S.FONT_LABEL)
ax.set_ylabel("3'UTR-length similarity (Pearson r)", fontsize=S.FONT_LABEL)
ax.set_title(
    "3'UTR-length vs expression state similarity\n"
    f"(above line = UTR more conserved: {frac_above:.0%} of pairs)",
    fontsize=S.FONT_LABEL,
)
ax.tick_params(labelsize=S.FONT_TICK)
fig.tight_layout()
S.save_and_show(fig, "transcriptome_similar_utrome_divergent_states", fig_dir=UNASSIGNED_DIR)

## 9. Robustness: full transcriptome (all genes) vs UTRome

Section 4 used matched expression from the UTR molecules themselves. Here we confirm the UTRome
structure also differs from the standard full transcriptome (all ~20k genes from the
gene-expression atlas, standard lognorm, genes expressed in >=10% of cells), i.e. the result is
not an artefact of restricting expression to the PAS-quantified genes.


In [ ]:
expr_ad = ad.read_h5ad(EXPR_H5)
raw_expr = expr_ad.raw.to_adata()
# reconcile cell IDs: expression BARCODE-batch -> BARCODE_BATCH (matches UTR obs_names)
ekey = pd.Index(
    [
        n.rsplit("-", 1)[0] + "_" + b
        for n, b in zip(raw_expr.obs_names, expr_ad.obs["batch"].astype(str))
    ]
)
raw_expr.obs_names = ekey
raw_expr = raw_expr[obs_names].copy()  # align to our 7-state cell set + order

Xe = raw_expr.X.tocsr() if issparse(raw_expr.X) else raw_expr.X
frac_expr = np.asarray((Xe > 0).mean(axis=0)).ravel()
expressed = frac_expr >= 0.10  # genes in >=10% of cells (Kremer criterion)
Xe = Xe[:, expressed]
expr_genes = raw_expr.var_names[expressed]
print(f"full-transcriptome genes (>=10% cells): {expressed.sum()}")

# per-state pseudobulk log-CPM
e_sums = np.vstack([np.asarray(Xe[m].sum(axis=0)).ravel() for m in ct_masks])
e_lib = e_sums.sum(axis=1, keepdims=True)
e_logcpm = np.log1p(e_sums / e_lib * 1e6).T  # (genes, states)

featFull = np.argsort(np.nanvar(e_logcpm, axis=1))[::-1][:TOP_N_FEATURES]
Cfull = pairwise_corr(e_logcpm[featFull])
Cfull_sp = pairwise_corr(e_logcpm[featFull], "spearman")

fixed_corr_figure(
    Cfull,
    Cw,
    "Full transcriptome (~20k genes)",
    "3'UTRome (UTR-length W)",
    "Pearson",
    "Robustness: full transcriptome vs 3'UTRome (Pearson)",
    "utrome_vs_full_transcriptome_corrmat_celltypes",
)

fixed_corr_figure(
    Cfull_sp,
    Cw_sp,
    "Full transcriptome (~20k genes)",
    "3'UTRome (UTR-length W)",
    "Spearman",
    "Robustness: full transcriptome vs 3'UTRome (Spearman)",
    "utrome_vs_full_transcriptome_corrmat_celltypes__spearman",
)

## 10. Pathway enrichment of the expression-invisible APA genes

We take the genes that are APA-dynamic (Fig-1 LRT, FDR<5%) yet expression-flat (|expression trend| <
0.3 along pseudotime), i.e. the genes whose regulation is invisible to the transcriptome (Section 7), and
test which biological pathways they are enriched for. Over-representation analysis (ORA) is run in R
(`apa-r`: clusterProfiler enrichGO + ReactomePA, native mouse Ensembl IDs). The primary
background is all tested multi-site genes (the Fig-1 ORA convention, so results are comparable and
powered); we also report a conservative coverage-matched background (well-covered genes only), which
guards against detectability artefacts. n is modest, so we show the top terms ranked by adjusted p and
mark the FDR<0.05 line; treat sub-threshold terms as suggestive, not established.


In [ ]:
# Foreground = APA-dynamic (Fig-1 LRT, FDR<5%) yet expression-flat (|expr trend|<0.3) - genes whose
# regulation is invisible to the transcriptome. Two backgrounds: (primary) all tested multi-site genes
# (the Fig-1 ORA convention, comparable + powered) and (robustness) the well-covered multi-site genes
# (coverage-matched, conservative). Native mouse Ensembl IDs (no ortholog mapping).
invis_fg = trend[trend["sig_apa"] & (trend["expr_trend"].abs() < FLAT)]["gene_id"].tolist()
bg_all = (
    rank["gene_id"].dropna().unique().tolist()
)  # all tested multi-site genes (Fig-1 convention)
bg_cov = list(np.array(genes)[covered_bin])  # well-covered genes (coverage-matched)
pd.DataFrame({"gene_id": invis_fg}).to_csv(RESULTS_DIR / "invisible_foreground.csv", index=False)
pd.DataFrame({"gene_id": bg_all}).to_csv(RESULTS_DIR / "invisible_bg_all.csv", index=False)
pd.DataFrame({"gene_id": bg_cov}).to_csv(RESULTS_DIR / "invisible_bg_cov.csv", index=False)
print(
    f"foreground (APA-dynamic & expression-flat): {len(invis_fg)} genes | "
    f"bg_all={len(bg_all)} | bg_cov={len(bg_cov)}"
)

# ORA in R (apa-r): clusterProfiler enrichGO(BP) for both backgrounds + ReactomePA for the
# primary. pvalueCutoff=1 returns all terms so Python can show the top ranks + the FDR line.
R_ORA = r"""
suppressMessages({library(clusterProfiler); library(org.Mm.eg.db); library(ReactomePA)})
a <- commandArgs(trailingOnly = TRUE)
fg <- read.csv(a[1])$gene_id
bg_all <- read.csv(a[2])$gene_id
bg_cov <- read.csv(a[3])$gene_id
outdir <- a[4]
go1 <- enrichGO(fg, OrgDb = org.Mm.eg.db, keyType = "ENSEMBL", universe = bg_all, ont = "BP",
                pAdjustMethod = "BH", pvalueCutoff = 1, qvalueCutoff = 1,
                minGSSize = 5, maxGSSize = 500, readable = TRUE)
write.csv(as.data.frame(go1), file.path(outdir, "invisible_ora_GO_BP.csv"), row.names = FALSE)
go2 <- enrichGO(fg, OrgDb = org.Mm.eg.db, keyType = "ENSEMBL", universe = bg_cov, ont = "BP",
                pAdjustMethod = "BH", pvalueCutoff = 1, qvalueCutoff = 1,
                minGSSize = 5, maxGSSize = 500, readable = TRUE)
write.csv(as.data.frame(go2), file.path(outdir, "invisible_ora_GO_BP_covbg.csv"), row.names = FALSE)
fg_e <- bitr(fg, "ENSEMBL", "ENTREZID", org.Mm.eg.db)$ENTREZID
bg_e <- bitr(bg_all, "ENSEMBL", "ENTREZID", org.Mm.eg.db)$ENTREZID
er <- enrichPathway(fg_e, universe = bg_e, organism = "mouse", pAdjustMethod = "BH",
                    pvalueCutoff = 1, qvalueCutoff = 1, minGSSize = 5, readable = TRUE)
write.csv(as.data.frame(er), file.path(outdir, "invisible_ora_Reactome.csv"), row.names = FALSE)
cat("ORA done")
"""
ORA_R = RESULTS_DIR / "run_invisible_ora.R"
ORA_R.write_text(R_ORA)
print("wrote", ORA_R)

In [ ]:
# Run the ORA in the apa-r env (subprocess; keeps this notebook single-kernel).
RSCRIPT = os.environ.get("M26_RSCRIPT", "Rscript")  # Rscript of the apa-r environment
proc = subprocess.run(
    [
        RSCRIPT,
        str(RESULTS_DIR / "run_invisible_ora.R"),
        str(RESULTS_DIR / "invisible_foreground.csv"),
        str(RESULTS_DIR / "invisible_bg_all.csv"),
        str(RESULTS_DIR / "invisible_bg_cov.csv"),
        str(RESULTS_DIR),
    ],
    capture_output=True,
    text=True,
    timeout=900,
)
print(proc.stdout[-400:] if proc.stdout else "(no stdout)")
if proc.returncode != 0:
    print("R FAILED; stderr tail:\n", proc.stderr[-1800:])

In [ ]:
def _read(p):
    return pd.read_csv(p) if p.exists() else pd.DataFrame()


go = _read(RESULTS_DIR / "invisible_ora_GO_BP.csv")  # primary: all-tested background
go_cov = _read(RESULTS_DIR / "invisible_ora_GO_BP_covbg.csv")  # robustness: coverage-matched
re_ = _read(RESULTS_DIR / "invisible_ora_Reactome.csv")


def _top(df, n=10):
    if df.empty or "p.adjust" not in df.columns:
        return pd.DataFrame()
    d = df.sort_values("p.adjust").head(n).iloc[::-1].copy()  # most-significant at top of barh
    d["Count"] = d["Count"].astype(int)
    return d


go_t, re_t = _top(go), _top(re_)
n_go_sig = int((go["p.adjust"] < 0.05).sum()) if not go.empty else 0
n_re_sig = int((re_["p.adjust"] < 0.05).sum()) if not re_.empty else 0
n_go_cov_sig = int((go_cov["p.adjust"] < 0.05).sum()) if not go_cov.empty else 0
print(
    f"GO BP FDR<0.05: {n_go_sig} (all-tested bg) / {n_go_cov_sig} (coverage-matched bg) | "
    f"Reactome FDR<0.05: {n_re_sig}"
)
for name, d in [("GO BP", go_t), ("Reactome", re_t)]:
    if not d.empty:
        print(f"\ntop {name} (most significant first):")
        print(d[["Description", "Count", "p.adjust"]].iloc[::-1].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(80) / 300))
for ax, d, ttl in [
    (axes[0], go_t, "GO Biological Process"),
    (axes[1], re_t, "Reactome"),
]:
    if d.empty:
        ax.text(0.5, 0.5, "no terms", ha="center", va="center", fontsize=S.FONT_LABEL)
        ax.axis("off")
        continue
    y = np.arange(len(d))
    vals = -np.log10(d["p.adjust"].clip(lower=1e-300))
    sig = (d["p.adjust"] < 0.05).values
    ax.barh(y, vals, color=np.where(sig, COL_UTR, "#B9A7CE"))
    ax.set_yticks(y)
    ax.set_yticklabels(d["Description"].str.slice(0, 44), fontsize=5)
    ax.axvline(-np.log10(0.05), color="k", lw=0.6, ls="--")
    ax.set_xlabel("-log10 adjusted p", fontsize=S.FONT_LABEL)
    ax.set_title(ttl, fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    for yi, cnt in zip(y, d["Count"]):
        ax.text(max(vals) * 0.02, yi, f"n={cnt}", va="center", fontsize=4.5)
fig.suptitle(
    f"Pathways of expression-invisible APA genes (foreground={len(invis_fg)}; all-tested bg; "
    "dashed = FDR 0.05)",
    fontsize=S.FONT_LABEL,
    y=1.02,
)
fig.tight_layout()
S.save_and_show(fig, "expression_invisible_apa_pathways", fig_dir=UNASSIGNED_DIR)

Result. Against the standard Fig-1 background (all tested multi-site genes) the
expression-invisible APA genes are enriched (Reactome, FDR<5%) for the same core machinery as the
whole APA-dynamic programme: cellular stress responses, cell-cycle / APC-C mitotic degradation, and
translation / RNA metabolism (GO BP shows the same RNA-splicing / translation / mRNA-processing themes
nominally, below FDR). Against the conservative coverage-matched background these terms are not
significant, so the enrichment is partly a detectability effect and we do not claim a distinct
pathway module. Interpretation: the 3'UTR regulation that is not visible in the transcriptome affects the same
stress / translation / RNA-processing genes as the overall APA-dynamic programme; it adds
post-transcriptional control of known players rather than a new pathway.


## 11. Summary and outputs

Panels written to `Fig1/unassigned_panels/` (SVG + PNG; Pearson = base name, Spearman = `__spearman`
variant), per-gene tables to `correlation_analyses_proportions/`. The final cell regenerates the
Figure-1 panel overview HTML so the new panels appear alongside the assigned ones.


In [ ]:
trend.to_csv(RESULTS_DIR / "expression_vs_utrlength_trends_per_gene.csv", index=False)
print("Panels saved to:", UNASSIGNED_DIR)
for p in sorted(UNASSIGNED_DIR.glob("*.png")):
    print("  -", p.name)
print("\nHeadline numbers -- does the 3'UTRome add information beyond the transcriptome?  YES:")
print(
    f"  cell-type Mantel (expr vs UTRome): r={mantel_r:.2f}, p={mantel_p:.3f}; "
    f"dendrogram coph. r={coph_r:.2f}"
)
print(f"  pseudotime-bin Mantel: r={mantel_r_b:.2f}, p={mantel_p_b:.3f}")
print(
    f"  [decoupling] per-gene median |Spearman(expr,W)| = {med_abs:.2f}; "
    f"|r|<0.3 for {frac_lo:.0%} of genes; {len(utr_only)} UTR-only genes "
    f"({int(utr_only['sig_apa'].sum())} APA-dynamic)"
)
print(
    f"  [invisible]  APA-dynamic but expression-flat: {len(invisible)} genes "
    f"({len(invisible) / max(int(apa_dyn.sum()), 1):.0%} of clearly-UTR-dynamic APA genes)"
)
print(
    f"  [direction]  3'UTR direction only weakly coupled to expression: OR={OR:.2f}, p={p_dir:.2f} "
    f"(expr-up leans shorten; many exceptions)"
)
print(
    f"  [timing]     UTR-vs-expression change offset: median {med_off:.0f} bins, "
    f"Wilcoxon p={w_p:.1e}; |offset|>=3 bins for {(both['offset'].abs() >= 3).mean():.0%}"
)
print(
    "\nInterpretation: the 3'UTR-length layer reorganises the cell-state/pseudotime structure, is "
    "decoupled gene-by-gene, regulates genes the transcriptome cannot see, and changes at a "
    "different pseudotime; its direction is only weakly coupled to expression -> overall it "
    "carries information the transcriptome alone does not."
)

if not re_t.empty:
    print(
        f"  [pathways]   invisible-APA genes: {n_re_sig} Reactome + {n_go_sig} GO-BP FDR<0.05 vs all-tested "
        f"bg (0/0 vs coverage-matched bg -> partly detectability); top: {re_t.iloc[-1]['Description']}"
    )